# Inventario del dataset TUSZ v2.0.6

Este notebook es el primer paso de la **Fase 1 (exploración de datos)** del TFM. Recorre todo el corpus TUSZ y construye un inventario: qué ficheros hay, cómo se registraron y dónde están las crisis.

No carga ninguna señal EEG. Solo lee la **cabecera** de cada EDF y los ficheros de anotaciones, así que tarda pocos minutos aunque el dataset ocupe unos 81 GB.

**Este notebook solo construye los datos.** El análisis (tablas resumen, elección del umbral de lead-in, figuras y conclusiones) está en `02_eda_tusz.ipynb`, que lee los CSV generados aquí. Así el inventario se ejecuta una vez y el EDA se puede repetir y ampliar sin volver a recorrer el dataset.

| Notebook | Qué hace | Salidas |
|---|---|---|
| `01_inventario_tusz.ipynb` | Recorre TUSZ y construye las tablas | `outputs/01_inventario/` |
| `02_eda_tusz.ipynb` | Analiza las tablas | `outputs/02_eda/` |

## Concepto clave: el *lead-in*

El **lead-in** de una crisis es el tiempo de señal **sin crisis** que hay justo antes de que empiece, contando solo dentro de su mismo fichero.

- Si es la **primera crisis del fichero**: desde el inicio del fichero hasta el inicio de la crisis.
- Si **hay otra crisis antes**: desde el final de la crisis anterior hasta el inicio de esta.

**Ejemplo** con un fichero de 20 min:

```
0 min ──── limpio ──── 6 min [CRISIS A] 7 min ── 2 min ── 9 min [CRISIS B] 10 min ──── 20 min
```

- Lead-in de A: **6 min** (de 0 a 6).
- Lead-in de B: **2 min** (de 7 a 9).

Con un umbral de 5 min, la crisis A sirve para predicción y la B no: antes de B solo hay 2 min, y además justo después de otra crisis, así que no son una señal "normal" previa a la crisis.

**Para qué sirve**: marca cuánto espacio hay para recortar la **ventana preictal**, que es el trozo de señal que el modelo aprenderá a reconocer como "se acerca una crisis". El lead-in viene dado por el fichero; la ventana preictal la elegimos nosotros y tiene que caber dentro. Si el lead-in es de 6 min, una ventana de 4 min más 1 min de margen cabe; si es de 2 min, no.

## Convención visual del TFM

Para saber de un vistazo con qué se trabaja, en todo el TFM se usan siempre los mismos iconos.

**Etiquetas de las anotaciones**

| Icono | Etiqueta | Significado |
|---|---|---|
| ⬜ | `bckg` | Fondo, sin crisis |
| ⬛ | `seiz` | Crisis, sin indicar el tipo (solo en `.csv_bi`). También se usa para `sin_tipo` |
| 🟥 | `fnsz` | Focal no específica |
| 🟦 | `gnsz` | Generalizada no específica |
| 🟧 | `cpsz` | Parcial compleja |
| 🟩 | `absz` | Ausencia |
| 🟪 | `tcsz` | Tónico-clónica |
| 🟨 | `tnsz` | Tónica |
| 🟫 | `spsz` | Parcial simple |
| 🔳 | `mysz` | Mioclónica |

`cnsz` (clónica) y `atsz` (atónica) no tienen icono porque no aparecen en TUSZ v2.0.6.

**Particiones del dataset**

| Icono | Partición | Uso |
|---|---|---|
| 🔵 | `train` | Entrenamiento |
| 🟡 | `dev` | Validación (ajuste de hiperparámetros) |
| 🟢 | `eval` | Test final (no se usa para ninguna decisión) |
| ⚪ | todas | Dataset completo |
| 🔴 | — | Reservado para avisos o problemas detectados |

> Los iconos vienen de `tfm_comun/iconos.py` y los colores de `tfm_comun/colores.py`, que comparten todos los notebooks del TFM. Si se cambia un icono o un color, se cambia solo allí.

En las figuras los iconos no se dibujan (matplotlib no los muestra bien), pero los colores de las particiones coinciden: azul 🔵, amarillo 🟡 y verde 🟢.

## 1. Importación de librerías

Solo se usan librerías estándar de análisis de datos. La cabecera EDF se lee a mano (son bytes de texto con un formato fijo), así que **no hace falta MNE** para este notebook.

In [10]:
from pathlib import Path
import json
import time

import pandas as pd

# Utilidades comunes a todo el TFM (paquete tfm_comun, un módulo por responsabilidad)
from tfm_comun.particiones import SPLITS
from tfm_comun.iconos import ICONO_ETIQUETA, ICONO_SPLIT, con_icono, tabla_con_iconos

## 2. Configuración general

Todas las rutas y parámetros se definen aquí. Si cambia la ubicación del dataset, solo hay que tocar esta celda.

- `RUTA_TUSZ`: carpeta `v2.0.6` del dataset. Se calcula a partir de la carpeta del notebook (`ALMA` está en `03.Codigo`, dos niveles por debajo de `TFM IA`).
- `RUTA_RESULTADOS`: `outputs/01_inventario`, junto al notebook. Aquí se guardan los CSV que usará el EDA.
- `EJEMPLO`: fichero que se usa para explicar cada paso. Es un fichero 🔵 `train` con tres crisis de 🟩 ausencia.

La configuración se guarda en `configuracion.json` para saber con qué parámetros y cuándo se generó el inventario.

In [11]:
# Rutas de entrada
RUTA_PROYECTO = Path.cwd().resolve().parents[1]          # D:\Dani\TFM IA
RUTA_TUSZ = RUTA_PROYECTO / '00.TUSZ' / 'v2.0.6'
RUTA_EDF = RUTA_TUSZ / 'edf'

# Rutas de salida
RUTA_RESULTADOS = Path('./outputs/01_inventario')
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

# Fichero de ejemplo: 🟩 crisis de ausencia (absz) del paciente aaaaabqv
EJEMPLO = 'aaaaabqv_s001_t000'

assert RUTA_EDF.exists(), f'No encuentro {RUTA_EDF}. Revisa RUTA_TUSZ.'
print(f'Dataset:    {RUTA_TUSZ}')
print(f'Resultados: {RUTA_RESULTADOS.resolve()}')

config = {
    'ruta_tusz': str(RUTA_TUSZ),
    'splits': SPLITS,
    'ejemplo': EJEMPLO,
    'fecha': time.strftime('%Y-%m-%d %H:%M'),
}
with open(RUTA_RESULTADOS / 'configuracion.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

Dataset:    D:\Dani\TFM IA\00.TUSZ\v2.0.6
Resultados: D:\Dani\TFM IA\03.Codigo\ALMA\outputs\01_inventario


## 3. Estructura de carpetas de TUSZ

Los ficheros están organizados así:

```
edf/{split}/{paciente}/{sesión}_{año}/{montaje}/{paciente}_{sesión}_{token}.edf
```

- **split**: 🔵 `train`, 🟡 `dev` o 🟢 `eval`. Cada paciente aparece en un solo split.
- **paciente**: identificador de 8 letras (por ejemplo `aaaaaaac`).
- **sesión**: `s001`, `s002`… Cada sesión es una visita distinta al hospital.
- **montaje**: referencia de los electrodos (`01_tcp_ar`, `02_tcp_le`, `03_tcp_ar_a`).
- **token**: `t000`, `t001`… Tramos de una misma sesión. Una grabación larga se parte en varios ficheros.

En este notebook, **fichero** significa una grabación continua (un token). Cada fichero se guarda en **tres archivos** con el mismo nombre:

| Extensión | Contenido | Uso en el TFM |
|---|---|---|
| `.edf` | Señal EEG cruda | Entrada de los modelos |
| `.csv` | Anotaciones por canal con el **tipo de crisis** | Fase 4 (clasificación) |
| `.csv_bi` | Anotaciones binarias `seiz` / `bckg` | Fase 3 (predicción) |

### Paciente de ejemplo: `aaaaabqv`

Para entender la jerarquía usamos el paciente de ejemplo. Tiene **2 sesiones**, y cada sesión tiene un fichero:

```
Paciente aaaaabqv  (🔵 train)
├── Sesión s001_2004  (montaje 02_tcp_le)
│   └── Fichero aaaaabqv_s001_t000   ← EJEMPLO
└── Sesión s002_2004  (montaje 02_tcp_le)
    └── Fichero aaaaabqv_s002_t001
```

Los niveles, de mayor a menor: **paciente → sesión → fichero → crisis**. Las crisis que hay dentro de cada fichero las veremos más adelante: en la sección 5 las del fichero de ejemplo, y en la sección 7 las de todo el dataset.

La siguiente celda lo comprueba en el disco: lista las sesiones y ficheros del paciente y los tres archivos del fichero de ejemplo.

In [12]:
paciente_ejemplo = EJEMPLO.split('_')[0]

# Sesiones y ficheros del paciente de ejemplo
print(f'Paciente {paciente_ejemplo}:')
for edf in sorted(RUTA_EDF.glob(f'*/{paciente_ejemplo}/*/*/*.edf')):
    split, _, sesion_carpeta, montaje = edf.relative_to(RUTA_EDF).parts[:4]
    marca = '  ← EJEMPLO' if edf.stem == EJEMPLO else ''
    print(f'  {con_icono(split)} | sesión {sesion_carpeta} | {montaje} | fichero {edf.stem}{marca}')
print()

# Fichero de ejemplo: lo buscamos sin escribir la ruta completa
ejemplo_edf = next(RUTA_EDF.glob(f'*/{paciente_ejemplo}/*/*/{EJEMPLO}.edf'))
print('Ruta relativa:', ejemplo_edf.relative_to(RUTA_EDF))
print('Archivos del fichero:')
for f in sorted(ejemplo_edf.parent.glob(ejemplo_edf.stem + '.*')):
    print(f'  {f.name:30} {f.stat().st_size / 1e6:8.2f} MB')

paciente, sesion, token = ejemplo_edf.stem.split('_')
print(f'\npaciente={paciente}  sesión={sesion}  token={token}  montaje={ejemplo_edf.parent.name}')

Paciente aaaaabqv:
  🔵 train | sesión s001_2004 | 02_tcp_le | fichero aaaaabqv_s001_t000  ← EJEMPLO
  🔵 train | sesión s002_2004 | 02_tcp_le | fichero aaaaabqv_s002_t001

Ruta relativa: train\aaaaabqv\s001_2004\02_tcp_le\aaaaabqv_s001_t000.edf
Archivos del fichero:
  aaaaabqv_s001_t000.csv             0.00 MB
  aaaaabqv_s001_t000.csv_bi          0.00 MB
  aaaaabqv_s001_t000.edf            23.72 MB

paciente=aaaaabqv  sesión=s001  token=t000  montaje=02_tcp_le


## 4. Lectura de la cabecera EDF

Un fichero EDF empieza con una cabecera de texto ASCII de longitud fija. Los campos que necesitamos son:

| Bytes | Campo |
|---|---|
| 236–244 | Número de bloques de datos (*data records*) |
| 244–252 | Duración de cada bloque (s) |
| 252–256 | Número de señales (`ns`) |
| 256 + ns·216 … | Muestras por bloque de cada señal (8 bytes por señal) |

Con eso calculamos:

- **duración** = nº de bloques × duración de cada bloque;
- **frecuencia de muestreo** = muestras por bloque / duración del bloque (se toma la primera señal).

Leer solo 256 + ns·256 bytes en lugar del fichero entero es lo que hace que el inventario sea rápido.

In [13]:
def leer_cabecera_edf(ruta):
    # Devuelve (nº de señales, frecuencia de muestreo de la 1ª señal, duración en segundos)
    with open(ruta, 'rb') as f:
        cabecera = f.read(256)
        n_registros = int(cabecera[236:244])
        duracion_registro = float(cabecera[244:252])
        n_senales = int(cabecera[252:256])

        # Saltamos los campos de cada señal que no necesitamos (etiqueta, transductor, unidades...)
        f.seek(256 + n_senales * 216)
        muestras_por_registro = [int(f.read(8)) for _ in range(n_senales)]

    fs = muestras_por_registro[0] / duracion_registro
    return n_senales, fs, n_registros * duracion_registro


n_senales, fs, duracion = leer_cabecera_edf(ejemplo_edf)
print(f'{ejemplo_edf.name}: {n_senales} señales, {fs:.0f} Hz, {duracion:.0f} s')

aaaaabqv_s001_t000.edf: 33 señales, 250 Hz, 1437 s


## 5. Lectura de las anotaciones `.csv` y `.csv_bi`

Cada EDF viene acompañado de dos ficheros de anotaciones. Los dos tienen el mismo formato: unas líneas de comentario que empiezan por `#` (entre ellas la duración del fichero) y después una tabla con las columnas `channel, start_time, stop_time, label, confidence`.

| Columna | Significado |
|---|---|
| `channel` | Canal del montaje TCP (por ejemplo `F7-T3`). En el `.csv_bi` vale siempre `TERM` = todo el fichero |
| `start_time`, `stop_time` | Inicio y fin del evento, en segundos desde el inicio del fichero |
| `label` | Etiqueta del evento (ver tabla siguiente) |
| `confidence` | Confianza del anotador (en TUSZ suele ser `1.0000`) |

**¿Qué es `TERM`?** No es un canal real. Indica que la anotación vale para **todos los canales a la vez**, es decir, para el fichero entero. Viene de cómo anota TUH (Temple University Hospital, autores de TUSZ), que usa dos formas de anotar:

| Tipo de anotación | Archivo | Columna `channel` | Qué dice |
|---|---|---|---|
| **Por canal** | `.csv` | `FP1-F7`, `F7-T3`… | En qué canales concretos se ve la crisis |
| **Global** (*term-based*) | `.csv_bi` | `TERM` | Que hay una crisis en el fichero, sin mirar canales |

En la práctica, `TERM` se puede ignorar: en el `.csv_bi` siempre vale lo mismo, y lo que importa son `start_time`, `stop_time` y `label`.

**¿Qué es `confidence`?** **No es la probabilidad de que haya una crisis.** Es la **seguridad del anotador** en la etiqueta de esa fila.

- El formato de anotación de TUH se usa tanto para anotaciones humanas como para las salidas de **detectores automáticos**. Un detector pondría aquí su probabilidad (por ejemplo `0.87` = "estoy seguro al 87 % de que esto es 🟩 `absz`").
- Un **neurólogo** que anota a mano pone `1.0000`: la etiqueta es definitiva.
- En TUSZ todas las anotaciones son **humanas y revisadas**, así que en los ficheros revisados vale siempre `1.0000`.

Para el TFM `confidence` no aporta información, porque no distingue unas crisis de otras. Las etiquetas de TUSZ son la **referencia** (*ground truth*) que el modelo intentará reproducir. Cuando nuestro modelo prediga, su salida sí será una probabilidad, y entonces tendría sentido usar este campo.

### 5.1. Etiquetas que aparecen en TUSZ

Las definiciones proceden de `DOCS/seizures_types_v02.xlsx`, traducidas al español.

**Etiquetas del `.csv_bi`** (binario: solo dice si hay crisis o no)

| Icono | Etiqueta | Nombre | Qué es |
|---|---|---|---|
| ⬜ | `bckg` | *Background* (fondo) | Actividad sin crisis: EEG normal, artefactos o cualquier cosa que no sea crisis |
| ⬛ | `seiz` | *Seizure* (crisis) | Hay una crisis, sin indicar de qué tipo |

**Etiquetas del `.csv`** (multiclase: dice el tipo de crisis y en qué canal se ve)

| Icono | Etiqueta | Nombre | Qué es |
|---|---|---|---|
| ⬜ | `bckg` | *Background* (fondo) | Actividad sin crisis en ese canal |
| 🟥 | `fnsz` | Focal no específica | Crisis que empieza en una zona concreta del cerebro, sin poder precisar más su tipo |
| 🟦 | `gnsz` | Generalizada no específica | Crisis que afecta a los dos hemisferios desde el principio, sin poder precisar más su tipo |
| 🟧 | `cpsz` | Parcial compleja | Crisis focal con **pérdida o alteración de la conciencia** (se clasifica por los signos clínicos) |
| 🟩 | `absz` | Ausencia | Crisis generalizada breve: el paciente "se desconecta" unos segundos (*petit mal*). En el EEG aparecen descargas punta-onda a ~3 Hz |
| 🟪 | `tcsz` | Tónico-clónica | Primero rigidez y después sacudidas (*grand mal*) |
| 🟨 | `tnsz` | Tónica | Rigidez del cuerpo durante la crisis |
| 🟫 | `spsz` | Parcial simple | Crisis focal en la que el paciente **conserva la conciencia** (se clasifica por los signos clínicos) |
| 🔳 | `mysz` | Mioclónica | Sacudidas bruscas y breves de las extremidades |
| — | `cnsz` | Clónica | Sacudidas rítmicas del cuerpo. Definida en TUSZ, pero no aparece en v2.0.6 |
| — | `atsz` | Atónica | Pérdida súbita del tono muscular. Definida en TUSZ, pero no aparece en v2.0.6 |

Relación entre los dos ficheros: cada ⬛ `seiz` del `.csv_bi` corresponde en el `.csv` a una de las etiquetas de color (🟥 🟦 🟧 🟩 …), repetida en cada canal donde se ve la crisis.

### 5.2. Ejemplo real: `aaaaabqv_s001_t000` (🔵 `train`)

Fichero de 1.437 s (~24 min) con **tres crisis de 🟩 ausencia**, de unos 7 s cada una.

**`.csv_bi`**: una fila por crisis, para todo el fichero.

| channel | start_time | stop_time | label | confidence |
|---|---|---|---|---|
| TERM | 179.8840 | 187.0160 | ⬛ seiz | 1.0000 |
| TERM | 242.6680 | 249.9280 | ⬛ seiz | 1.0000 |
| TERM | 286.0160 | 293.5840 | ⬛ seiz | 1.0000 |

**`.csv`**: una fila por canal y crisis (extracto de la primera crisis).

| channel | start_time | stop_time | label | confidence |
|---|---|---|---|---|
| FP1-F7 | 179.8840 | 187.0160 | 🟩 absz | 1.0000 |
| F7-T3 | 179.8840 | 187.0160 | 🟩 absz | 1.0000 |
| T3-T5 | 179.8840 | 187.0160 | 🟩 absz | 1.0000 |
| FP2-F8 | 179.8840 | 187.0160 | 🟩 absz | 1.0000 |
| CZ-C4 | 179.8840 | 187.0160 | 🟩 absz | 1.0000 |
| … (17 canales más, todos 🟩 `absz`) | | | | |

El fichero completo tiene 66 filas: 3 crisis × 22 canales.

Cómo se lee:

- Cada ⬛ `seiz` del `.csv_bi` aparece en el `.csv` como 🟩 `absz` en **los 22 canales**: la ausencia es una crisis **generalizada**, afecta a todo el cerebro a la vez.
- El `.csv_bi` solo dice que hay tres crisis y cuándo; el tipo está en el `.csv`.
- Las tres crisis están muy juntas: entre la 1.ª y la 2.ª pasan 56 s y entre la 2.ª y la 3.ª, 36 s.

Por eso usamos el `.csv_bi` para la **predicción** (Fase 3: ¿viene una crisis?) y el `.csv` para la **clasificación** (Fase 4: ¿de qué tipo es?).

> 🔴 **Aviso para la Fase 3.** El lead-in de estas crisis es de 3,0 min, 0,9 min y 0,6 min (ver concepto al inicio del notebook). Ninguna llega al umbral de 5 min, así que este fichero **no sirve para predicción**, aunque sí para clasificación. En todo TUSZ solo hay 8 ausencias con lead-in ≥ 5 min (6 en 🔵 `train`, 2 en 🟡 `dev`, 0 en 🟢 `eval`). Las ausencias son breves y van en racimos: encajan mejor en la clasificación (Fase 4) que en la predicción.

### 5.3. Lectura en código

La función `leer_anotaciones` lee la duración de la cabecera (para comprobar después que coincide con la del EDF) y la tabla de anotaciones. La probamos con el fichero de ejemplo.

In [14]:
def leer_anotaciones(ruta):
    # Devuelve (duración declarada en la cabecera, DataFrame de anotaciones)
    duracion = None
    with open(ruta) as f:
        for linea in f:
            if linea.startswith('# duration'):
                duracion = float(linea.split('=')[1].split()[0])
            if not linea.startswith('#'):
                break
    return duracion, pd.read_csv(ruta, comment='#')


dur_csv, anot_multi = leer_anotaciones(ejemplo_edf.with_suffix('.csv'))
dur_bi, anot_bi = leer_anotaciones(ejemplo_edf.with_suffix('.csv_bi'))
print(f'Duración en .csv: {dur_csv} s | en .csv_bi: {dur_bi} s | en .edf: {duracion} s')
print('\n.csv_bi:')
display(anot_bi.assign(label=anot_bi.label.map(con_icono)))
print('.csv (primeras filas):')
display(anot_multi.head().assign(label=anot_multi.head().label.map(con_icono)))

Duración en .csv: 1437.0 s | en .csv_bi: 1437.0 s | en .edf: 1437.0 s

.csv_bi:


,channel,start_time,stop_time,label,confidence
0,TERM,179.884,187.016,⬛ seiz,1.0
1,TERM,242.668,249.928,⬛ seiz,1.0
2,TERM,286.016,293.584,⬛ seiz,1.0


.csv (primeras filas):


,channel,start_time,stop_time,label,confidence
0,FP1-F7,179.884,187.016,🟩 absz,1.0
1,FP1-F7,242.668,249.928,🟩 absz,1.0
2,FP1-F7,286.016,293.584,🟩 absz,1.0
3,F7-T3,179.884,187.016,🟩 absz,1.0
4,F7-T3,242.676,249.928,🟩 absz,1.0


## 6. Tipo dominante de cada crisis

Para predicción usamos los eventos del `.csv_bi` (una fila = una crisis). Pero el `.csv_bi` no dice de qué **tipo** es la crisis; eso está en el `.csv`, canal a canal.

Para asignar un tipo a cada evento:

1. Se toman las filas del `.csv` que no son `bckg`.
2. Se calcula cuánto se solapa cada una con el intervalo del evento.
3. Se suma el solapamiento por etiqueta (sobre todos los canales).
4. La etiqueta con más solapamiento es el **tipo dominante**.

Si ninguna etiqueta se solapa, el evento queda como `sin_tipo`. Casi nunca ocurre, pero conviene que quede registrado en lugar de descartarlo sin avisar.

In [15]:
def tipo_dominante(anot_multi, inicio, fin):
    crisis = anot_multi[anot_multi.label != 'bckg']
    solape = (crisis.stop_time.clip(upper=fin) - crisis.start_time.clip(lower=inicio)).clip(lower=0)
    solape_por_tipo = solape.groupby(crisis.label).sum()
    solape_por_tipo = solape_por_tipo[solape_por_tipo > 0]
    return solape_por_tipo.idxmax() if len(solape_por_tipo) else 'sin_tipo'


# Ejemplo: tipo de cada crisis del fichero de ejemplo
for _, ev in anot_bi[anot_bi.label == 'seiz'].iterrows():
    print(f'{ev.start_time:8.1f} - {ev.stop_time:8.1f} s  ->  {con_icono(tipo_dominante(anot_multi, ev.start_time, ev.stop_time))}')

   179.9 -    187.0 s  ->  🟩 absz
   242.7 -    249.9 s  ->  🟩 absz
   286.0 -    293.6 s  ->  🟩 absz


## 7. Recorrido completo del dataset → `edfs.csv`

Aplicamos las funciones anteriores a los 8.140 EDF. Por cada fichero guardamos una fila con:

| Columna | Significado |
|---|---|
| `split`, `patient`, `session`, `token`, `montage`, `file` | Identificación del fichero |
| `n_signals`, `fs` | Número de señales y frecuencia de muestreo |
| `edf_dur_s`, `csv_dur_s` | Duración según el EDF y según el CSV |
| `n_seiz_events_bi`, `seiz_secs_bi` | Número de crisis y segundos en crisis según el `.csv_bi` |
| `labels` | Tipos de crisis presentes en el `.csv`, separados por `;` |

A la vez guardamos en una lista cada crisis del `.csv_bi` con su tipo dominante; la usaremos en la sección 8.

In [16]:
filas_ficheros, filas_crisis = [], []
inicio_t = time.time()

for split in SPLITS:
    edfs = sorted((RUTA_EDF / split).glob('*/*/*/*.edf'))
    print(f'{con_icono(split)}: {len(edfs)} EDF')

    for i, edf in enumerate(edfs, 1):
        paciente, sesion, token = edf.stem.split('_')
        n_senales, fs, dur_edf = leer_cabecera_edf(edf)
        dur_csv, anot_multi = leer_anotaciones(edf.with_suffix('.csv'))
        _, anot_bi = leer_anotaciones(edf.with_suffix('.csv_bi'))
        eventos = anot_bi[anot_bi.label == 'seiz']
        tipos = sorted(set(anot_multi.label) - {'bckg'})

        filas_ficheros.append({
            'split': split, 'patient': paciente, 'session': sesion, 'token': token,
            'montage': edf.parent.name, 'file': edf.stem,
            'n_signals': n_senales, 'fs': fs,
            'edf_dur_s': dur_edf, 'csv_dur_s': dur_csv,
            'n_seiz_events_bi': len(eventos),
            'seiz_secs_bi': round((eventos.stop_time - eventos.start_time).sum(), 2),
            'labels': ';'.join(tipos) if tipos else None,
        })
        for _, ev in eventos.iterrows():
            filas_crisis.append({
                'split': split, 'patient': paciente, 'session': sesion, 'token': token,
                'file': edf.stem, 'label': tipo_dominante(anot_multi, ev.start_time, ev.stop_time),
                'start': ev.start_time, 'stop': ev.stop_time, 'file_dur': dur_edf,
            })

        if i % 1000 == 0:
            print(f'  {i}/{len(edfs)} ({time.time() - inicio_t:.0f} s)')

df_ficheros = pd.DataFrame(filas_ficheros)
df_ficheros.to_csv(RUTA_RESULTADOS / 'edfs.csv', index=False)
print(f'\n{len(df_ficheros)} ficheros inventariados en {time.time() - inicio_t:.0f} s')
df_ficheros.head()

🔵 train: 5057 EDF
  1000/5057 (6 s)
  2000/5057 (12 s)
  3000/5057 (17 s)
  4000/5057 (23 s)
  5000/5057 (27 s)
🟡 dev: 2203 EDF
  1000/2203 (31 s)
  2000/2203 (35 s)
🟢 eval: 880 EDF

8140 ficheros inventariados en 39 s


,split,patient,session,token,montage,file,n_signals,fs,edf_dur_s,csv_dur_s,n_seiz_events_bi,seiz_secs_bi,labels
0,train,aaaaaaac,s001,t000,02_tcp_le,aaaaaaac_s001_t000,33,250.0,301.0,301.0,1,200.32,cpsz
1,train,aaaaaaac,s001,t001,02_tcp_le,aaaaaaac_s001_t001,33,250.0,236.0,236.0,1,188.25,cpsz
2,train,aaaaaaac,s002,t000,02_tcp_le,aaaaaaac_s002_t000,33,250.0,262.0,262.0,1,202.02,cpsz
3,train,aaaaaaac,s004,t000,02_tcp_le,aaaaaaac_s004_t000,33,250.0,321.0,321.0,0,0.00,NaN
4,train,aaaaaaac,s004,t002,02_tcp_le,aaaaaaac_s004_t002,33,250.0,626.0,626.0,0,0.00,NaN


## 8. Tabla de crisis y su *lead-in* → `crisis.csv`

Para la predicción hay que distinguir dos conceptos:

- **Lead-in**: tiempo sin crisis que hay **antes** de cada crisis. Es un dato del fichero; no lo elegimos.
- **Ventana preictal**: trozo de señal que recortaremos antes de la crisis para entrenar el modelo. La elegimos nosotros y **tiene que caber dentro del lead-in**.

**El lead-in se mide solo dentro del mismo fichero.** Una sesión puede estar partida en varios tokens (`t000`, `t001`…), pero no sabemos si son seguidos:

- las cabeceras EDF están anonimizadas (todas empiezan el 1 de enero a las `00.00.00`), así que no hay hora real;
- ni el `AAREADME.txt` ni `DOCS/metadata_v06r.xlsx` indican si hay pausas entre tokens;
- en 108 sesiones faltan tokens en la numeración (por ejemplo `t000` y `t002` sin `t001`).

Encadenar tokens inflaría el lead-in con una suposición que no podemos comprobar. Dentro de un fichero, en cambio, la señal es continua con seguridad.

Cálculo: el lead-in es el tiempo desde el final de la crisis anterior **del mismo fichero** hasta el inicio de esta. Si es la primera crisis del fichero, es el tiempo desde el inicio del fichero.

In [17]:
df_crisis = pd.DataFrame(filas_crisis)

# Lead-in: inicio de la crisis menos el final de la crisis anterior del mismo fichero
df_crisis = df_crisis.sort_values(['split', 'patient', 'file', 'start']).reset_index(drop=True)
fin_anterior = df_crisis.groupby('file').stop.shift().fillna(0)   # 0 = inicio del fichero
df_crisis['leadin_s'] = (df_crisis.start - fin_anterior).round(1)
df_crisis['primera_del_fichero'] = df_crisis.groupby('file').cumcount() == 0

df_crisis.to_csv(RUTA_RESULTADOS / 'crisis.csv', index=False)
print(f'{len(df_crisis)} crisis guardadas')
print(f'Crisis que son la primera de su fichero: {df_crisis.primera_del_fichero.mean():.0%}')
print(f'Lead-in máximo posible: {df_crisis.leadin_s.max() / 60:.1f} min (limitado por la duración del fichero)')
df_crisis.head()

2664 crisis guardadas
Crisis que son la primera de su fichero: 41%
Lead-in máximo posible: 26.8 min (limitado por la duración del fichero)


,split,patient,session,token,file,label,start,stop,file_dur,leadin_s,primera_del_fichero
0,dev,aaaaaajy,s001,t000,aaaaaajy_s001_t000,fnsz,14.2442,57.9846,1750.0,14.2,True
1,dev,aaaaaajy,s001,t000,aaaaaajy_s001_t000,gnsz,57.9964,196.0107,1750.0,0.0,False
2,dev,aaaaaajy,s001,t000,aaaaaajy_s001_t000,fnsz,254.0130,362.9638,1750.0,58.0,False
3,dev,aaaaaajy,s001,t000,aaaaaajy_s001_t000,fnsz,477.0717,651.9384,1750.0,114.1,False
4,dev,aaaaaajy,s001,t000,aaaaaajy_s001_t000,fnsz,712.0451,908.0735,1750.0,60.1,False


## 9. Comprobaciones de calidad

Antes de sacar conclusiones verificamos tres cosas:

1. **Ningún paciente aparece en dos splits.** Si apareciera, el modelo vería en test datos de un paciente con el que ha entrenado y los resultados serían optimistas (*data leakage*).
2. **La duración del EDF coincide con la del CSV.** Si no coincide, las anotaciones podrían estar desplazadas respecto a la señal.
3. **Los eventos del `.csv_bi` y del inventario cuadran** (mismo número de crisis en las dos tablas).

In [18]:
pacientes = {s: set(df_ficheros[df_ficheros.split == s].patient) for s in SPLITS}
compartidos = {
    'train/dev': len(pacientes['train'] & pacientes['dev']),
    'train/eval': len(pacientes['train'] & pacientes['eval']),
    'dev/eval': len(pacientes['dev'] & pacientes['eval']),
}
print('1. Pacientes compartidos entre splits (debe ser 0):', compartidos)

dif_duracion = (df_ficheros.edf_dur_s - df_ficheros.csv_dur_s).abs()
print(f'2. Ficheros con duración EDF ≠ CSV (> 1 s): {(dif_duracion > 1).sum()}')

print(f'3. Crisis en edfs.csv: {df_ficheros.n_seiz_events_bi.sum()} | en crisis.csv: {len(df_crisis)}')
print(f'   Crisis sin tipo asignado: {(df_crisis.label == "sin_tipo").sum()}')

1. Pacientes compartidos entre splits (debe ser 0): {'train/dev': 0, 'train/eval': 0, 'dev/eval': 0}
2. Ficheros con duración EDF ≠ CSV (> 1 s): 0
3. Crisis en edfs.csv: 2664 | en crisis.csv: 2664
   Crisis sin tipo asignado: 1


## 10. Archivos generados y siguiente paso

En `outputs/01_inventario/`:

- `configuracion.json`: rutas, fichero de ejemplo y fecha de generación.
- `edfs.csv`: una fila por EDF (identificación, montaje, canales, frecuencia, duración, nº de crisis y tipos).
- `crisis.csv`: una fila por crisis, con su tipo dominante, su lead-in dentro del fichero y si es la primera del fichero.

Los CSV se guardan **sin iconos** para que se puedan leer fácilmente desde código.

**Siguiente paso**: abrir `02_eda_tusz.ipynb`, que analiza estas dos tablas.